In [27]:
using LowLevelFEM, LinearAlgebra, SparseArrays
using IterativeSolvers, Preconditioners

In [28]:
spectral_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "spectral.jl"
)

@assert isfile(spectral_path) "Copy spectral.jl into the LowLevelFEM src directory first: $spectral_path"

Base.include(
    LowLevelFEM,
    spectral_path
)

using LowLevelFEM: spectralTransformation

┌ Warning: Replacing docs for `LowLevelFEM._sem_legendre_pair :: Tuple{Integer, Real}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_gll :: Tuple{Integer}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_barycentric_weights :: Tuple{AbstractVector}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_lagrange_matrix :: Tuple{AbstractVector, AbstractVector}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_finalize_column_pattern! :: Tuple{Vector{Vector{Int64}}}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_csc_from_column_pattern :: Tuple{Integer, Integer, Vector{Vector{Int64}}}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_set_unique! :: Tuple{SparseMatrixCS

In [29]:
openGeometry("plate-with-hole.geo")
#openPreProcessor()

In [30]:
mat = Material("body")
P1 = Field([mat], type=:VectorField, dim=2, fieldName=:u)
P2 = Field([mat], type=:VectorField, dim=2, fieldName=:u, basis=:spectral);

In [31]:
K1 = ∫(SymGrad(P1) ⋅ D(:PlaneStress, mat) ⋅ SymGrad(P1))
f1 = ∫(P1 ⋅ [1,0], Γ="right")

K2 = ∫(SymGrad(P2) ⋅ D(:PlaneStress, mat) ⋅ SymGrad(P2))
f2 = ∫(P2 ⋅ [1,0], Γ="right");

In [32]:
bc = BoundaryCondition("left", ux=0, uy=0);

In [33]:
@time u1 = solveField(Symmetric(K1), f1, support=[bc])
@time u2 = solveField(Symmetric(K2), f2, support=[bc])

err = norm(DoFs(u1) - DoFs(u2)) / norm(DoFs(u1))

  0.654770 seconds (100.50 k allocations: 304.041 MiB, 19.16% gc time, 40.11% compilation time: 100% of which was recompilation)
  0.515501 seconds (31.64 k allocations: 925.706 MiB, 22.72% gc time)


6.107344881599066e-13

In [34]:
@time u1 = solveField(Symmetric(K1), f1, support=[bc], solver=:cg)
@time u2 = solveField(Symmetric(K2), f2, support=[bc], solver=:cg)

err = norm(DoFs(u1) - DoFs(u2)) / norm(DoFs(u1))

  1.725137 seconds (38.57 k allocations: 212.427 MiB, 0.62% gc time, 0.78% compilation time: 100% of which was recompilation)
  1.538632 seconds (34.88 k allocations: 838.009 MiB, 2.52% gc time)


1.4918489399022e-9

In [35]:
p1 = LowLevelFEM.prepare_singlefield_system(
    Symmetric(K1), f1, [bc]
)

p2 = LowLevelFEM.prepare_singlefield_system(
    Symmetric(K2), f2, [bc]
)

@time x1 = LowLevelFEM.solve_linear_system(
    p1.A, p1.B;
    solver=:cg
)

@time x2 = LowLevelFEM.solve_linear_system(
    p2.A, p2.B;
    solver=:cg
);

  1.318822 seconds (3.47 k allocations: 77.820 MiB, 1.69% gc time)
  1.461277 seconds (3.35 k allocations: 77.798 MiB, 4.81% gc time)


In [36]:
tol = sqrt(eps())

@time x1, hist1 = cg(
    p1.A,
    @view(p1.B[:, 1]);
    reltol=tol,
    maxiter=size(p1.A, 1),
    log=true
)

@time x2, hist2 = cg(
    p2.A,
    @view(p2.B[:, 1]);
    reltol=tol,
    maxiter=size(p2.A, 1),
    log=true
)

@show hist1.iters
@show hist2.iters

 18.149272 seconds (22.12 k allocations: 1.651 MiB)
 12.317783 seconds (12.98 k allocations: 1.455 MiB)
hist1.iters = 4511
hist2.iters = 2689


2689

In [37]:
A1 = p1.A isa Symmetric ? parent(p1.A) : p1.A
A2 = p2.A isa Symmetric ? parent(p2.A) : p2.A

@show size(A1)
@show size(A2)

@show nnz(A1)
@show nnz(A2)
@show nnz(A2) / nnz(A1)

size(A1) = (30774, 30774)
size(A2) = (30774, 30774)
nnz(A1) = 2168156
nnz(A2) = 2168156
nnz(A2) / nnz(A1) = 1.0


1.0

In [38]:
Ts, Rs = LowLevelFEM.field_transformation(P2)

(sparse([2403, 3185, 3189, 3203, 3207, 3209, 3211, 3213, 3215, 5765  …  30848, 30852, 30839, 30841, 30843, 30845, 30840, 30842, 30844, 30846], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  30854, 30854, 30855, 30855, 30855, 30855, 30856, 30856, 30856, 30856], [1.0, -0.11718750000000006, 0.039062500000000014, 0.039062500000000014, -0.11718750000000006, 0.013732910156250014, -0.0045776367187500035, 0.001525878906250001, -0.0045776367187500035, 0.039062500000000014  …  -0.11840671663579501, 0.8840317166357948, 0.014020150544469454, -0.10467529296874999, 0.7815120760180302, -0.10467529296874999, 0.014020150544469454, -0.10467529296874999, 0.7815120760180302, -0.10467529296874999], 30856, 30856), sparse([14105, 14107, 14111, 14125, 14127, 14131, 14135, 14137, 14141, 14106  …  30855, 30840, 30842, 30844, 30846, 30848, 30850, 30852, 30854, 30856], [1, 1, 1, 1, 1, 1, 1, 1, 1, 2  …  30855, 30856, 30856, 30856, 30856, 30856, 30856, 30856, 30856, 30856], [-0.026914552635302895, -0.0034707760655282383, 0.0007

In [39]:
M = ∫(P2 ⋅ P2)
M[:,:]

30856×30856 SparseMatrixCSC{Float64, Int64} with 1088496 stored entries:
⎡⢿⣷⡗⣿⡏⠀⠱⡦⡟⠟⣿⠗⠓⠛⠙⢻⡾⠻⢓⢰⣧⠽⣷⢴⣴⣵⣮⣶⣷⣾⣾⣶⣶⣶⡶⣧⠁⠀⠀⠀⎤
⎢⣽⣭⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣯⎥
⎢⠋⠉⣿⣿⣿⣿⣝⣿⢟⢿⠻⡿⡯⡯⣛⣿⣿⣿⣽⣿⣫⢛⣹⣏⣫⠯⢿⢟⣏⠋⣍⠫⢝⢍⢩⣩⡯⣹⣙⣉⎥
⎢⠱⡦⣿⣿⣷⣽⣻⣾⣼⡯⢟⣽⣿⢭⢡⡶⡮⣾⠯⡸⣛⣫⣟⣟⢦⢼⣛⣗⡊⡩⠲⠒⠤⣃⠧⣐⢸⠤⡯⠏⎥
⎢⣿⠍⣿⣿⣿⣕⡶⡿⣿⣿⡟⣻⣹⣾⣛⡓⡾⣿⢋⣿⣴⢯⣈⣵⠦⡢⢟⡻⣧⣿⣀⠈⡅⠷⣈⡽⡼⣬⢙⡳⎥
⎢⢿⠟⣿⣿⣿⡦⣟⣵⣿⣩⡿⣯⣧⣿⢽⢾⣿⣚⡷⣅⣿⢦⣺⣯⡿⡿⣗⡕⣜⣘⡴⠉⡆⠆⠿⣞⢅⠋⢋⣏⎥
⎢⣽⠀⣿⣿⡯⡯⡟⣟⣳⣾⣭⣿⣿⣿⣯⣟⢿⡿⣾⢿⢏⢿⣸⣶⣥⠲⣙⣏⡪⣄⠫⠈⡃⡢⣴⣸⣝⣛⣊⠃⎥
⎢⣷⣀⣿⣿⣿⣼⢡⡶⢿⠸⣳⣗⣯⢿⢿⣷⣟⣿⣿⠿⡹⠿⡛⡽⠸⡱⣹⣫⣆⣚⡊⢘⠊⡪⣕⣆⣧⢐⠖⡄⎥
⎢⣾⡋⣿⣿⣿⣿⣪⣯⣾⣯⣻⢻⣿⡷⣿⣽⣻⣾⣦⣽⡼⢧⣭⣦⢗⣿⣮⣗⣏⡻⣪⢹⠟⢙⢽⢻⣹⣐⡿⣵⎥
⎢⢙⣐⣿⣿⣷⣿⣋⡣⣯⣴⠝⢯⣾⣟⣿⡟⣌⣿⡿⣯⣿⣽⣷⡿⢴⢿⣟⡼⣣⠦⣹⣟⣳⡸⢿⡆⣛⠶⠋⣯⎥
⎢⣍⡟⣿⣿⣯⢚⡿⣸⡴⣟⠻⣟⣯⣕⣷⡎⠶⣏⣟⣿⡿⣯⣷⣿⣷⡿⣟⣋⢙⣛⢉⠔⠦⠼⢧⣹⢟⠧⡾⡖⎥
⎢⢙⣟⣿⣿⡷⢾⣿⢽⢆⣼⡾⣾⢲⣾⣟⡬⠣⣿⣽⡿⣽⣿⣻⣾⡿⣯⣿⡯⢿⡼⡜⠽⢑⣛⣵⣫⣣⡥⣔⣂⎥
⎢⢔⣿⣿⣿⡯⡞⣈⣗⠨⡣⣿⡯⢡⡛⢖⡢⣽⣵⣴⣗⣽⡿⡿⣯⣿⣿⣻⣿⣽⣿⢸⡽⢷⣎⣇⣺⡯⣶⠇⠀⎥
⎢⢪⣿⣿⣿⣿⢗⢿⢼⣿⡱⢝⠽⡷⢼⡷⣺⢮⢿⣛⡽⡿⢹⡿⡿⣿⣾⣿⣿⣜⣷⣡⡎⢀⣙⣙⢻⣿⣻⡿⠻⎥
⎢⣹⣿⣿⣿⡯⠙⡎⡨⣭⣿⣒⢹⠊⢮⣨⢹⣯⡹⠩⡞⣷⢰⣛⡷⣷⣿⢶⣽⣿⣿⣿⣼⣝⣝⡿⡿⢭⠗⣯⠮⎥
⎢⢺⣿⣿⣿⡧⡙⢸⠂⡀⠘⡔⠋⡋⠂⣊⢈⣎⣚⣷⢾⢃⠔⣖⡍⣖⡶⡡⠾⣛⣿⢿⣷⣷⣺⢬⡇⡿⢓⣼⡇⎥
⎢⢸⣿⣿⣿⡗⢕⠤⢣⢥⡍⠨⠍⠩⡨⡪⡠⣟⢁⣙⡺⣈⡇⣵⢰⡹⢷⣄⢰⣗⢽⣹⣻⣿⣿⣿⣯⣉⣿⣿⣿⎥
⎢⠼⣯⣿⣿⡇⣲⢉⢣⣆⡼⣻⢧⣐⣻⠱⢽⣷⣓⠻⠷⣍⣳⡵⣻⣩⣹⣷⣘⣿⡯⠦⠷⡿⣿⣿⣿⣿⣿⡺⡷⎥
⎢⠁⠀⣿⣿⣏⣫⠒⡖⡒⣯⡥⠑⣷⢹⢉⢛⢓⢺⢻⡜⠿⡕⠍⡾⢫⣯⣿⣻⢧⠗⢿⢋⣧⣼⣿⣿⣿⣿⣿⣷⎥
⎣⠀⠀⡿⣿⡗⢸⡯⠏⢷⡰⡯⢴⠮⠘⠘⠥⢟⣯⡯⣤⢺⠯⠰⢹⠉⠁⣿⡋⡫⡟⠶⠿⣿⣿⢾⡮⢿⣿⣿⣿⎦

In [40]:
M = ∫(P2 ⋅ P2, gauss=:gll)
droptol!(M.A, 1e-10)
Ms = Ts' * M[:,:] * Ts
droptol!(Ms, 1e-13)
#dropzeros!(Ms)
(Ms)[1000:1008, 1000:1008]

9×9 SparseMatrixCSC{Float64, Int64} with 9 stored entries:
 7.75436   ⋅        ⋅        ⋅       …   ⋅        ⋅        ⋅        ⋅ 
  ⋅       2.62609   ⋅        ⋅           ⋅        ⋅        ⋅        ⋅ 
  ⋅        ⋅       2.62609   ⋅           ⋅        ⋅        ⋅        ⋅ 
  ⋅        ⋅        ⋅       2.17086      ⋅        ⋅        ⋅        ⋅ 
  ⋅        ⋅        ⋅        ⋅           ⋅        ⋅        ⋅        ⋅ 
  ⋅        ⋅        ⋅        ⋅       …  5.99539   ⋅        ⋅        ⋅ 
  ⋅        ⋅        ⋅        ⋅           ⋅       5.99539   ⋅        ⋅ 
  ⋅        ⋅        ⋅        ⋅           ⋅        ⋅       7.44837   ⋅ 
  ⋅        ⋅        ⋅        ⋅           ⋅        ⋅        ⋅       7.44837

In [41]:
Ms

30856×30856 SparseMatrixCSC{Float64, Int64} with 30856 stored entries:
⎡⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⎦

In [42]:
@show nnz(Ts)
@show nnz(Rs)

@show Base.summarysize(K2.A) / 2^20
@show Base.summarysize(Ts) / 2^20
@show Base.summarysize(Rs) / 2^20

@show (Base.summarysize(Ts) + Base.summarysize(Rs)) /
      Base.summarysize(K2.A)

nnz(Ts) = 273240
nnz(Rs) = 273240
Base.summarysize(K2.A) / 2 ^ 20 = 33.453834533691406
Base.summarysize(Ts) / 2 ^ 20 = 4.404884338378906
Base.summarysize(Rs) / 2 ^ 20 = 4.404884338378906
(Base.summarysize(Ts) + Base.summarysize(Rs)) / Base.summarysize(K2.A) = 0.2633410728413056


0.2633410728413056

In [43]:
P2 = Field([mat];
    type=:VectorField,
    dim=2,
    fieldName=:u,
    basis=:spectral
)

T, R = spectralTransformation(P2; check=true)

@show T === P2.approximation.T
@show R === P2.approximation.R
@show norm(R*T - I, Inf)

M = ∫(P2 ⋅ P2; gauss=:gll)
Ms = T' * M.A * T
droptol!(Ms, 1e-14)

@show nnz(Ms)
@show size(Ms, 1)

T === P2.approximation.T = true
R === P2.approximation.R = true
norm(R * T - I, Inf) = 4.884981308350689e-15
nnz(Ms) = 32110
size(Ms, 1) = 30856


30856

In [44]:
saveField("test", u1)
u3 = loadField("test")

@show u3.model.basis
@show u3.model.reducedOrder
@show u3.model.approximation.T
@show u3.model.approximation.R
@show norm(u3.a - u1.a)

u3.model.basis = :lagrange
u3.model.reducedOrder = false
u3.model.approximation.T = nothing
u3.model.approximation.R = nothing
norm(u3.a - u1.a) = 0.0


0.0

In [45]:
@time begin
    AT = K2.A * T
    Kr1 = T' * AT
end

@time begin
    TA = T' * K2.A
    Kr2 = TA * T
end

@show norm(Kr1 - Kr2) / norm(Kr1)

@show nnz(AT)
@show nnz(TA)

@show Base.summarysize(AT) / 2^20
@show Base.summarysize(TA) / 2^20

  0.240719 seconds (50 allocations: 646.530 MiB, 42.51% gc time)
  0.151246 seconds (46 allocations: 646.529 MiB, 13.90% gc time)
norm(Kr1 - Kr2) / norm(Kr1) = 1.8638532999658644e-16
nnz(AT) = 2176992
nnz(TA) = 2176992
Base.summarysize(AT) / 2 ^ 20 = 321.02709197998047
Base.summarysize(TA) / 2 ^ 20 = 321.02709197998047


321.02709197998047

In [46]:
@time prep = LowLevelFEM.prepare_singlefield_system(
    Symmetric(K2),
    f2,
    [bc]
)

  0.389101 seconds (31.52 k allocations: 759.505 MiB, 53.58% gc time)


(A = [416515.41067032376 1215.2440250713794 … 0.0 0.0; 1215.2440250713794 369307.54542964604 … 0.0 0.0; … ; 0.0 0.0 … 905638.2713239068 -19083.65812420844; 0.0 0.0 … -19083.65812420844 915531.800327242], B = [0.0; 0.0; … ; 0.0; 0.0;;], T = sparse([2403, 3185, 3189, 3203, 3207, 3209, 3211, 3213, 3215, 5765  …  30848, 30852, 30839, 30841, 30843, 30845, 30840, 30842, 30844, 30846], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  30854, 30854, 30855, 30855, 30855, 30855, 30856, 30856, 30856, 30856], [1.0, -0.11718750000000006, 0.039062500000000014, 0.039062500000000014, -0.11718750000000006, 0.013732910156250014, -0.0045776367187500035, 0.001525878906250001, -0.0045776367187500035, 0.039062500000000014  …  -0.11840671663579501, 0.8840317166357948, 0.014020150544469454, -0.10467529296874999, 0.7815120760180302, -0.10467529296874999, 0.014020150544469454, -0.10467529296874999, 0.7815120760180302, -0.10467529296874999], 30856, 30856), Q = nothing, template = VectorField(Matrix{Float64}[], [0.0; 0.0; … ; 0.

In [47]:
@time X = LowLevelFEM.solve_linear_system(
    prep.A,
    prep.B;
    solver=:cg
)

  1.507890 seconds (3.35 k allocations: 77.798 MiB)


30774×1 Matrix{Float64}:
  0.0003548819207093553
 -7.828610427493789e-5
  0.00035415444344378654
 -7.957390404453985e-5
  0.00035260995639721586
 -8.21734493520794e-5
  0.0003508305716805474
 -8.49977102324616e-5
  0.0003497891842760336
 -8.65857386840357e-5
  0.0003589020303163532
 -8.011865490824418e-5
  0.00035801713467696264
  ⋮
  0.0006652760676847425
  0.00017061639365604928
  0.0006720545378726881
  0.00016936746813442353
  0.0006788111341031469
  0.00016801800799325252
  0.0006675642582929298
  0.0001693284709611163
  0.0006742503609679923
  0.00016819097733756163
  0.0006809098686658825
  0.00016697141494583482

In [48]:
function csc_storage(A)
    return (
        colptr = Base.summarysize(A.colptr) / 2^20,
        rowval = Base.summarysize(A.rowval) / 2^20,
        nzval  = Base.summarysize(A.nzval)  / 2^20,
        total  = (
            Base.summarysize(A.colptr) +
            Base.summarysize(A.rowval) +
            Base.summarysize(A.nzval)
        ) / 2^20
    )
end

@show size(K2.A)
@show nnz(K2.A)
@show csc_storage(K2.A)

@show size(T)
@show nnz(T)
@show csc_storage(T)

@show size(AT)
@show nnz(AT)
@show csc_storage(AT)

@show size(Kr1)
@show nnz(Kr1)
@show csc_storage(Kr1)

@show size(prep.A)
#@show nnz(prep.A)
@show csc_storage(parent(prep.A))

size(K2.A) = (30856, 30856)
nnz(K2.A) = 2176992
csc_storage(K2.A) = (colptr = 0.2354583740234375, rowval = 16.609169006347656, nzval = 16.609169006347656, total = 33.45379638671875)
size(T) = (30856, 30856)
nnz(T) = 273240
csc_storage(T) = (colptr = 0.2354583740234375, rowval = 2.0846939086914062, nzval = 2.0846939086914062, total = 4.40484619140625)
size(AT) = (30856, 30856)
nnz(AT) = 2176992
csc_storage(AT) = (colptr = 0.2354583740234375, rowval = 160.3957977294922, nzval = 160.3957977294922, total = 321.0270538330078)
size(Kr1) = (30856, 30856)
nnz(Kr1) = 2176992
csc_storage(Kr1) = (colptr = 0.2354583740234375, rowval = 160.3957977294922, nzval = 160.3957977294922, total = 321.0270538330078)
size(prep.A) = (30774, 30774)
csc_storage(parent(prep.A)) = (colptr = 0.234832763671875, rowval = 16.54175567626953, nzval = 16.54175567626953, total = 33.31834411621094)


(colptr = 0.234832763671875, rowval = 16.54175567626953, nzval = 16.54175567626953, total = 33.31834411621094)

In [49]:
AT2 = copy(AT)

@show nnz(AT2)
@show Base.summarysize(AT)  / 2^20
@show Base.summarysize(AT2) / 2^20
@show csc_storage(AT)
@show csc_storage(AT2)

nnz(AT2) = 2176992
Base.summarysize(AT) / 2 ^ 20 = 321.02709197998047
Base.summarysize(AT2) / 2 ^ 20 = 33.453834533691406
csc_storage(AT) = (colptr = 0.2354583740234375, rowval = 160.3957977294922, nzval = 160.3957977294922, total = 321.0270538330078)
csc_storage(AT2) = (colptr = 0.2354583740234375, rowval = 16.609169006347656, nzval = 16.609169006347656, total = 33.45379638671875)


(colptr = 0.2354583740234375, rowval = 16.609169006347656, nzval = 16.609169006347656, total = 33.45379638671875)

In [50]:
Kr3 = copy(Kr1)

@show Base.summarysize(Kr1) / 2^20
@show Base.summarysize(Kr3) / 2^20

Base.summarysize(Kr1) / 2 ^ 20 = 321.02709197998047
Base.summarysize(Kr3) / 2 ^ 20 = 33.453834533691406


33.453834533691406

In [51]:
AT = K2.A * T

@show Base.summarysize(AT) / 2^20

sizehint!(AT.rowval, length(AT.rowval); shrink=true)
sizehint!(AT.nzval,  length(AT.nzval);  shrink=true)

@show Base.summarysize(AT) / 2^20

Base.summarysize(AT) / 2 ^ 20 = 321.02709197998047
Base.summarysize(AT) / 2 ^ 20 = 33.453834533691406


33.453834533691406

In [52]:
Kr = T' * AT

@show Base.summarysize(Kr) / 2^20

sizehint!(Kr.rowval, length(Kr.rowval); shrink=true)
sizehint!(Kr.nzval,  length(Kr.nzval);  shrink=true)

@show Base.summarysize(Kr) / 2^20

Base.summarysize(Kr) / 2 ^ 20 = 321.02709197998047
Base.summarysize(Kr) / 2 ^ 20 = 33.453834533691406


33.453834533691406